# Mt. Edgecumbe volcanic InSAR time series analysis with HyP3 and MintPy

## Background
In this notebook, you will learn how to perform a time series InSAR analysis using the popular Small BAseline Subset (SBAS) technique. SBAS is well suited for monitoring slow, consistent deformation of the earth's surface caused by processes such as slow landslides, subsidence, fault creep and volcanic inflation. In this tutorial we will focus on a volcanic inflation use case at [Mt. Edgecumbe, Alaska](https://en.wikipedia.org/wiki/Mount_Edgecumbe_(Alaska)).

In April 2022, a seismic swarm near Mt. Edgecumbe in southeast Alaska suggested there could be renewed activity at the volcano. However, since the last sign of activity at Mt. Edgecumbe was about 800 years ago, there was no monitoring equipment installed at the volcano. This made it difficult to determine if the seismic swarm was unrelated to volcanic activity, or if the volcano was returning to an active state. Luckily, the team at the Alaska Volcano Observatory (AVO) included experienced InSAR users, and they were able use a time series InSAR analysis to show that volcanic inflation had begun at the site in 2018. This analysis took much less time than traditional methods, and was critical for providing timely information about the status of the volcano to the neighboring city of Sitka, Alaska.

This notebook will show you how to perform the InSAR time series analysis that the AVO team used to reach their conclusions. It follows the methodology detailed in their recent paper [(Grapenthin et al., 2022)](https://doi.org/10.1029/2022GL099464), which utilizes On Demand InSAR products from the Alaska Satellite Facility (ASF) and the MintPy time series analysis software.

## Analysis Techniques
For their analysis, Grapenthin et al. utilized a distributed scatterer (DS) time-series InSAR analysis. This methodology is appropriate for several reasons:

1. InSAR is one of the only remote sensing techniques sensitive enough to measure the small amount of deformation occurring at Mt. Edgecumbe
2. InSAR analyses in natural areas are plagued many erroneous or irrelevant deformation signals. By utilizing a time-series analysis, we can deal with these confounding signals by combining information from many interferogram.
3. Since this is a natural setting with few scatterers that remain highly correlated through time (e.g., many areas experience snow-coverage in winter), we will be using using a distributed scatter (DS) time-series approach. This technique sacrifices spatial resolution to increase the coherence of the InSAR measurements.

In the future, this analysis would be a good candidate for a combined persistent scatterer and distributed scatterer analysis (PS+DS), but a reliable open-source PS+DS software package was not available when Grapenthin et al. conducted this analysis.

## Notebook Structure
In this notebook we will:

1. Use the [HyP3 Python SDK](https://hyp3-docs.asf.alaska.edu/using/sdk/) to:
   - Request On Demand InSAR products from ASF HyP3
   - Download the InSAR products when they are done processing
   - Prep these products for MintPy by subsetting them to a common region
2. Use [MintPy](https://mintpy.readthedocs.io/en/latest/) and various Python utilities to:
   - Assess the quality of the interferogram network we will use for the analysis
   - Assess whether the data quality is suitable for a time series analysis
   - Perform SBAS time series analyses for three InSAR stacks over Mt. Edgecumbe
   - Demonstrate how to view the results of a time series analysis
   - Reproduce the main InSAR figure from the AVO team's recent publication [(Figure 2 from Grapenthin et al., 2022)](https://doi.org/10.1029/2022GL099464)

---

**Note:** This notebook uses staged data to ensure that there is adequate time to step through the InSAR time series workflow and discuss the analysis methods. It also assumes that you have some familiarity with InSAR processing already. If you're new to InSAR and working with ASF's on-demand services, you may find the following resources useful:
1. ASF's [InSAR On Demand StoryMap](https://storymaps.arcgis.com/stories/68a8a3253900411185ae9eb6bb5283d3)
2. The `data_prep.py` script in the same directory as this notebook
3. [OpenSARlab](https://opensarlab-docs.asf.alaska.edu/)'s highly detailed walkthrough of preparing ASF's On-Demand (HyP3) data for MintPy using these notebooks:
    - [Prepare a HyP3 InSAR Stack for MintPy](https://nbviewer.org/github/ASFOpenSARlab/opensarlab-notebooks/blob/master/SAR_Training/English/Master/Prepare_HyP3_InSAR_Stack_for_MintPy.ipynb)
    - [MintPy Time-series Analysis](https://nbviewer.org/github/ASFOpenSARlab/opensarlab-notebooks/blob/master/SAR_Training/English/Master/MintPy_Time_Series_From_Prepared_Data_Stack.ipynb)

---

This notebook and the ones listed above assume that you're working in OpenSARlab. However, you can also run these 
notebooks outside of OpenSARlab by creating [this conda environment](https://github.com/ASFOpenSARlab/opensarlab-envs/blob/main/Environment_Configs/osl_mintpy_env.yaml) and launching these notebooks from within this environment.

## 0. Initial Setup

To run this notebook, you'll need to be in the `osl_mintpy` conda environment within OpenSARLab.

Alternatively, you can set up your own environment by running these commands in your shell (you'll need to have [conda](https://docs.conda.io/projects/continuumio-conda/en/latest/user-guide/install/index.html) installed):
```shell
curl -OL https://github.com/ASFOpenSARlab/opensarlab-envs/blob/main/Environment_Configs/osl_mintpy_env.yaml
conda env create -f osl_mintpy_env.yml
```
Then launch this notebook from the new environment:
```shell
conda activate osl_mintpy
jupyter lab eusar_mtedgecumbe_ts_analysis.ipynb
```

Once you have completed the setup for one of these two environments, you are ready to start working with the data.

We'll also download and unzip the [ERA5 weather files](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5) MintPy uses for its atmospheric correction. MintPy will download these files itself from the ERA5 data repository if they are not present, but this will take significantly longer than downloading the same files from our staged bucket.

In [ ]:
import boto3
import os
import zipfile
from pathlib import Path
from botocore import UNSIGNED
from botocore.client import Config

file = 'weather.zip'
s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))

download_path = Path(file)
s3_client.download_file('ffwilliams2-shenanigans', f'ERA5/{file}', download_path)
with zipfile.ZipFile(download_path, 'r') as zip_ref:
    zip_ref.extractall('.')
os.unlink(download_path)

## 1. Request On Demand InSAR products from ASF HyP3

A major step towards working with SAR data at scale is learning how to request and download data programmatically. Accomplishing these tasks via code makes it much easier to request large quantities of data and to make similar requests in the future.

To request the generation of an Interferometric Synthetic Aperture Radar (InSAR) product from ASF, you can follow the general steps outlined in the cells below. For this example, we'll be requesting the generation of an interferogram that shows inflation-related displacement at Mt. Edgecumbe:

1. **Create an account**: If you don't already have NASA Earthdata Login credentials, create a [NASA Earthdata login](https://urs.earthdata.nasa.gov/) profile. This will allow you connect to ASF HyP3 via the Python SDK. Running the cell below will prompt you for your Earthdata username and password

In [ ]:
import hyp3_sdk as sdk

hyp3 = sdk.HyP3(prompt=True)

2. **Define project variables**: Next, we'll define a project name, and create some directories to store the data we create. The default name is `edgecumbe_descending_174`, since we are looking at Mt. Edgecumbe, and processing Sentinel-1 data that comes from the descending pass of the satellite constellation with 174th relative orbit. We also provide a path to the scene pair csv we will be using. To aditionally process data for the ascending pass, you create a new project name, and redefine `scene_pair_path` variable to `ascending_50_pairs.csv` and `ascending_79_pairs.csv`.

In [ ]:
from pathlib import Path

# Possible options for project_name are:
#     edgecumbe_descending_174
#     edgecumbe_ascending_50
#     edgecumbe_ascending_79
project_name = 'edgecumbe_descending_174'

# Possible options for scene_pair_path are:
#     edgecumbe_pairs/descending_174_pairs.csv
#     edgecumbe_pairs/ascending_50_pairs.csv
#     edgecumbe_pairs/ascending_79_pairs.csv
scene_pair_path = Path('edgecumbe_pairs/descending_174_pairs.csv')

In [ ]:
from pathlib import Path

project_dir = Path(project_name)
project_dir.mkdir(exist_ok=True)
data_dir = project_dir / 'hyp3'
data_dir.mkdir(exist_ok=True)

3. **Load the pair information**: For this tutorial, we will be using scene pair information provided by the AVO team on the [Zenodo page for their publication](https://zenodo.org/records/7151431). When running your own analyses, we recommend using ASF's [Vertex](https://search.asf.alaska.edu/) data search portal to find Sentinel-1 scenes you would like to process.

In [ ]:
import pandas as pd

pairs = pd.read_csv(scene_pair_path)
pairs

In total we will be creating 127 interferograms for the descending pass!

4. **Define your processing parameters**: Next we'll set the specific processing parameters for our InSAR products. This includes selecting the number of looks (which determines the pixel spacing of the output product), and whether to apply a water mask before phase unwrapping. If you are using the InSAR products for MintPy analysis, you will also need to include the DEM and look vector products with your output product. You can read more about the available parameters within the [HyP3 SDK documentation](https://hyp3-docs.asf.alaska.edu/using/sdk_api/#hyp3_sdk.hyp3.HyP3.submit_insar_job).

In [ ]:
from datetime import datetime

batch_name = f'{project_name}_{datetime.now().strftime("%Y%m%d")}'
opts = {'name': batch_name, 'looks': '20x4', 'include_dem':True, 'include_look_vectors':True, 'apply_water_mask': True}
opts

5. **Submit a processing request**: Once you have defined your parameters, use the HyP3 SDK's [`submit_insar_job`](https://hyp3-docs.asf.alaska.edu/using/sdk_api/#hyp3_sdk.hyp3.HyP3.submit_insar_job) function to submit a processing request. You can use the `project_name` name argument to group sets of requests together under one name so that you can easily look them up later.

In [ ]:
jobs = sdk.Batch()
for idx, reference, secondary in pairs[['scene1', 'scene2']].itertuples():
    jobs += hyp3.submit_insar_job(reference, secondary, **opts)

6. **Monitor the processing status**: After submitting your request, ASF will process your data using HyP3, and output an InSAR product. You can monitor the status of your request by calling the [`watch`](https://hyp3-docs.asf.alaska.edu/using/sdk_api/#hyp3_sdk.hyp3.HyP3.watch) HyP3 SDK method.

In [ ]:
jobs = hyp3.watch(jobs)

7. **Find your InSAR products later**: You can also find your products later (they expire in two weeks!) by using the project name to find your jobs

In [ ]:
jobs = hyp3.find_jobs(name=batch_name)
print(jobs)

<div class="alert alert-success">
8. <b>Download pre-processed data</b>: For this tutorial, Forrest Williams has already processed the data for us. We can find his jobs by running:
</div>

In [ ]:
# jobs = hyp3.find_jobs(user_id='ffwilliams2', name='edgecumbe_desending_174_20240415')
# jobs = hyp3.find_jobs(user_id='ffwilliams2', name='edgecumbe_ascending_50_20240415')
# jobs = hyp3.find_jobs(user_id='ffwilliams2', name='edgecumbe_ascending_79_20240415')
jobs = hyp3.find_jobs(user_id='ffwilliams2', name='edgecumbe_desending_174_20240415')
print(jobs)

...then using the `download_files` method to download the data. The HyP3 SDK also includes an `extract_zipped_product` utility that you can use to unzip the files.

In [ ]:
insar_products = jobs.download_files(data_dir)
insar_products = [sdk.util.extract_zipped_product(ii) for ii in insar_products]

Now you have all of your requested HyP3 products on your local computer!

## 3. Prepare Datasets for MintPy

Before we can load our HyP3 products into our time series analysis software (MintPy), there is one more step we'll need to do. MintPy expects that all input products have the same spatial extents. Currently our HyP3-generated interferograms have slightly different extents depending on which input Sentinel-1 scenes we used, so we'll need to subset all of our HyP3 products to a common overlap.

To do this we'll use two functions, which are defined in the cell below. The first function will find the common spatial extent for all of our HyP3 products, and the second will clip our HyP3 products to this extent.

In [ ]:
from pathlib import Path
from typing import List, Union
from osgeo import gdal

gdal.UseExceptions()

def get_common_overlap(file_list: List[Union[str, Path]]) -> List[float]:
    """Get the common overlap of  a list of GeoTIFF files
    
    Arg:
        file_list: a list of GeoTIFF files
    
    Returns:
         [ulx, uly, lrx, lry], the upper-left x, upper-left y, lower-right x, and lower-right y
         corner coordinates of the common overlap
    """
    
    corners = [gdal.Info(str(dem), format='json')['cornerCoordinates'] for dem in file_list]

    ulx = max(corner['upperLeft'][0] for corner in corners)
    uly = min(corner['upperLeft'][1] for corner in corners)
    lrx = min(corner['lowerRight'][0] for corner in corners)
    lry = max(corner['lowerRight'][1] for corner in corners)
    return [ulx, uly, lrx, lry]


def clip_hyp3_products_to_common_overlap(data_dir: Union[str, Path], overlap: List[float]) -> None:
    """Clip all GeoTIFF files to their common overlap
    
    Args:
        data_dir:
            directory containing the GeoTIFF files to clip
        overlap:
            a list of the upper-left x, upper-left y, lower-right-x, and lower-tight y
            corner coordinates of the common overlap
    Returns: None
    """

    
    files_for_mintpy = ['_water_mask.tif', '_corr.tif', '_unw_phase.tif', '_dem.tif', '_lv_theta.tif', '_lv_phi.tif']

    for extension in files_for_mintpy:

        for file in data_dir.rglob(f'*{extension}'):

            dst_file = file.parent / f'{file.stem}_clipped{file.suffix}'

            gdal.Translate(destName=str(dst_file), srcDS=str(file), projWin=overlap)

Now that we've defined these functions, let's use them to subset our products to a common overlap.

In [ ]:
files = data_dir.glob('*/*_dem.tif')

overlap = get_common_overlap(files)
clip_hyp3_products_to_common_overlap(data_dir, overlap)

We're now ready to begin our time-series analysis!

## 3. Time-series Analysis with MintPy

Let's go ahead and download the staged data from Amazon Web Services cloud storage solution, the Simple Storage Service (S3). This cell will download the data from S3, then unzip it into your working directory.

### 3.1 MintPy Background

The main interface for MintPy is the command line interface (CLI) tool `smallbaselineApp.py`, which you can call from this notebook or in your terminal. Whenever you begin working with a new CLI tool, it's a good idea to read the introductory documentation for the tool. This will save a ton of headaches in the long run! MintPy's documentation can be found on its [GitHub page](https://github.com/insarlab/MintPy), or by calling the tool with the `--help` flag (see below). Most Python CLI tools have a `--help` option, so get in the habit of using it whenever you begin working with a new tool.

In [ ]:
!smallbaselineApp.py --help

MintPy has a ton of configuration options that we will use to produce the best analysis that we can, and to replicate the workflow of Grapenthin et al.. We'll be using the defaults for many of MintPy's settings (you can see all of MintPy's defaults by running the command `smallbaselineApp.py -H`), but we will set some key settings ourselves. Run the command below to view the settings for one of our InSAR stacks. Other than the data loading options, we use the same settings for all stacks.

### 3.2 Create our MintPy configuration

You can control many aspects of the MintPy analysis by specifying options in a MintPy configuration file (see the cell below). First we'll create our configuration file, then we'll discuss a few of the most relevant options.

In [ ]:
roi = [6315473, 6340109, 446628, 466268]
reference_point = [6330696, 456350]
mintpy_config = project_dir / 'mintpy_config.txt'
text = f"""##---------processor:
mintpy.load.processor        = hyp3
mintpy.plot                  = no

##---------interferogram datasets:
mintpy.load.unwFile          = {data_dir.name}/*/*_unw_phase_clipped.tif
mintpy.load.corFile          = {data_dir.name}/*/*_corr_clipped.tif

##---------geometry datasets:
mintpy.load.demFile          = {data_dir.name}/*/*_dem_clipped.tif
mintpy.load.incAngleFile     = {data_dir.name}/*/*_lv_theta_clipped.tif
mintpy.load.azAngleFile      = {data_dir.name}/*/*_lv_phi_clipped.tif
mintpy.load.waterMaskFile    = {data_dir.name}/*/*_water_mask_clipped.tif

##--------dataset geographic subset
mintpy.subset.lalo           = [{roi[0]}:{roi[1]}, {roi[2]}:{roi[3]}]
mintpy.reference.lalo        = {reference_point}

##--------network selections
mintpy.network.coherenceBased  = yes
mintpy.network.minCoherence    = 0.7

##--------unwrapping error correction
mintpy.unwrapError.method      = no

##--------troposperic phase correction
mintpy.troposphericDelay.method = pyaps # pyaps eventually
mintpy.troposphericDelay.weatherModel = ERA5
mintpy.troposphericDelay.weatherDir   = ../weather

##--------topographic phase correction
mintpy.topographicResidual     = yes
"""

mintpy_config.write_text(text)

These settings can be broken down into six groups: input data options, geographic options, inteferogram network options, unwrapping error correction options, tropospheric correction options, and topographic residual correction options:

1. **Input Data Options:**
    - The settings in the first three sections (processor, interferogram datasets, and geometry datasets) tell MintPy which InSAR processor we used to create our data, and where to find various datasets it needs for processing.
2. **Geographic Options:**
    -  The next section gives MintPy some geographic information. It tells MintPy to only load a subset of the data surrounding Mt. Edgecumbe, and to use the point `6330696, 456350` as the reference point for the time series velocity calculation. Both of these options are specified in the geographic projection of the input data, which in this case is UTM zone 8N (EPSG:32608).
3. **Interferogram Network Options:**
    - An important adage to remember when conducting InSAR analyses is *"garbage in, garbage out"*. If you include poor quality (highly decorrelated) interferograms in your analysis, they will lead to poor results that are difficult to interpret. For this reason, we exclude any interferograms that have an average coherence less than `0.7`. **THIS IS THE MOST IMPORTANT PARAMETER YOU WILL SET WHEN USING MINTPY!!!** Take the time to make sure this value is right for your use case. In addition, MintPy gives you the option to remove individual problematic interferograms using the `mintpy.network.excludeIfgIndex` option. In most cases it's worth taking the time to go through each interferogram individually and remove any that have obvious decorrelation or unwrapping issues.
4. **Unwrapping Error Correction Options:**
    - Unwrapping error correction uses phase closure trends to reduce the number of unwrapping errors in the input data (read more [here](https://doi.org/10.1016/j.cageo.2019.104331)), and is turned on by default. This process requires a connected components layer, which is not available for products generated using HyP3, so we will be skipping this correction.
5. **Tropospheric Phase Correction Options:**
    - SAR signals propagate through the troposphere at slightly different rates depending on atmospheric conditions, which can lead to troposphere-related signals in interferograms. We need to correct for these tropospheric impacts, which can mask actual deformation signals or appear to indicate deformation where none exists. You can learn more about this error source [here](https://doi.org/10.1016/j.earscirev.2019.03.008). Most troposphere correction algorithms utilize external atmospheric data to predict the error caused by this effect, then correct for it. In this case, we're using the [PyAPS](https://github.com/insarlab/PyAPS) Python package in conjunction with [ERA5](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5) data to perform this correction. 
6. **Topographic Residual Correction Options:**
    - Errors in the DEMs used for InSAR processing can lead to errors that are proportional to the perpendicular baseline of the InSAR pair. This is another effect that is important to correct for. MintPy uses the method proposed by [Heresh and Amelung, 2013](https://doi.org/10.1109/TGRS.2012.2227761) to perform this correction.

### 3.3 Run MintPy Analysis

Now that we've covered the basics of our MintPy configuration, it's time to perform our analysis! As mentioned above, we will be processing data from three stacks, so you'll need to run the section of code starting here at section 3.3 to just before section 4 three times, once for each data stack. Make sure to run each stack in order by modifying the stack variable in the cell below.

#### 3.3.1 Load data

First we'll load our HyP3 products into MintPy. MintPy stores it's intermediate data in HDF5 files, so we'll use MintPy's `load_data` step to convert our individual interferogram GeoTIFFs into a interferogram stack HDF5 file.

In [ ]:
!smallbaselineApp.py $mintpy_config --dir $project_dir --dostep load_data

#### 3.3.2 Check Network

As we mentioned before, selecting the right interferogram network is the most important way to ensure a high quality time series InSAR analysis. Because of this, we're going to look at various ways you can analyze the quality of your interferogram network.

##### 3.3.1.1 Network Connectivity

The first thing to check is that you have a fully connected network (e.g., every date in your analysis window is covered by at least one interferogram). This can sometimes be a challenge, due to decorrelation caused by seasonal differences. In snowy areas like Mt. Edgecumbe, the presence and/or condition of snow on the ground introduces significant decorrelation effects that hinder the accurate measurement of surface displacements. As a result, interferograms generated during winter periods tend to be unreliable. To overcome this challenge, it becomes necessary to exclude interferograms from the winter season. However, excluding winter interferograms leaves a data gap that can hamper your connectivity. To address this issue, you can create longer baseline interferograms, which span a longer period of time and cover the missing winter period. By including a few interferograms with long temporal baselines, you can ensure connectivity while also not including any highly decorrelated interferograms. See the graph below to see the network that Grapenthin et al. used.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.collections import LineCollection
from mintpy.utils.network import get_date12_list

plt.close()

ifgram_stack = project_dir / 'inputs' / 'ifgramStack.h5'

pair_list = pd.DataFrame([pair.split('_') for pair in get_date12_list(ifgram_stack)], columns=['date1', 'date2'])
pair_list['date1'] = pd.to_datetime(pair_list['date1'])
pair_list['date2'] = pd.to_datetime(pair_list['date2'])

# subplot 1 data
lines = []
bridges = []
for i, row in pair_list.iterrows():
    point1 = [mdates.date2num(row['date1']), i]
    point2 = [mdates.date2num(row['date2']), i]
    if point2[0] - point1[0] > 48:
        bridges.append([point1, point2])
    else:
        lines.append([point1, point2])

# subplot 2 data
date1_number = mdates.date2num(pair_list.date1)
date2_number = mdates.date2num(pair_list.date2)
dates = np.concatenate((date1_number, date2_number))
min_date = np.min(dates)
max_date = np.max(dates)

date_range = np.arange(np.min(dates), np.max(dates))
coverage = np.zeros(int(np.max(dates) - np.min(dates)))
for ifg_date1, ifg_date2 in zip(date1_number, date2_number):
    ifg_coverage = np.arange(ifg_date1 - min_date, ifg_date2 - min_date, dtype=int)
    coverage[ifg_coverage] += 1


f, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
line_segments = LineCollection(lines, array=range(len(lines)), linewidths=1, cmap='gist_rainbow', label='Interferograms')
bridge_segments = LineCollection(bridges, color='black', linestyle='dashed', linewidths=1, label='Bridging Interferograms')
ax1.add_collection(line_segments)
ax1.add_collection(bridge_segments)
ax1.set(
    ylabel='Interferogram Number',
    xlabel='Date',
    xlim=(mdates.date2num(np.min(pair_list['date1'])) - 10, mdates.date2num(np.max(pair_list['date2'])) + 10),
    ylim=(-1, pair_list.shape[0] + 1),
)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%b'))
ax1.legend(loc='upper left')

ax2.plot(date_range, coverage, color='black', linewidth=2)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%b'))
ax2.set(xlabel='Date', ylabel='# of interferograms covering date', ylim=(0, np.max(coverage) + 1))

plt.tight_layout()

Each line in the top graph represents an interferogram. You can see that most interferograms span a relatively small date window, but that there are a few long temporal baseline interferograms that span the winter snow season. The bottom graph is a histogram displaying how many interferograms cover each date. As you can see, we have at least one interferogram for each date, so our network is fully connected!

##### 3.3.1.2 Network Quality

The next thing we want to consider is network quality. Setting an average coherence threshold of `0.7` already excludes a lot of interferograms (the interferograms whose numbers are red in the graph below are excluded), but there are still some we could consider removing to avoid including interferograms with unwrapping errors. For instance, take a look at interferogram number 12 in the `ascending_50` stack.

In [ ]:
from mintpy.cli import view

ifgram_stack = project_dir / 'inputs' / 'ifgramStack.h5'
view.main(f'{ifgram_stack} --noverbose'.split())

If we wanted to remove this interferogram we could run MintPy's `modify_network.py` utility as shown below.

In [ ]:
# !modify_network.py {stack}/inputs/ifgramStack.h5 --exclude-ifg-index 12

Then, if we wanted to reset our network for any reason, we could use this command.

In [ ]:
# !modify_network.py {stack}/inputs/ifgramStack.h5 --reset

##### 3.3.1.3 View final network

Now that we've made all of our modifications, let's run `smallbaselineApp.py`'s `modify_network` step and view the results

In [ ]:
!smallbaselineApp.py --dir $project_dir $mintpy_config --dostep modify_network

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from mintpy.cli import plot_network

plt.close()

ifgram_stack = project_dir / 'inputs' / 'ifgramStack.h5'
coh_avg = project_dir / 'coherenceSpatialAvg.txt'
plot_network.main(f'{coh_avg} -t {mintpy_config} --show-kept --figsize 12 3'.split())

#### 3.3.2 Check Reference Point

The next important thing to check is that we've selected an appropriate reference point. Since all deformation in our time series will be relative to this point, it is very important that this point is highly coherent and is in a non-deforming location. Note that if you are using multiple co-located stacks (like we are) you should use the same reference point for all stacks. First, we need to run the `reference_point` step:

In [ ]:
!smallbaselineApp.py $mintpy_config --dir $project_dir  --dostep reference_point

Then we can view the location of the reference point on top of the average spatial coherence:

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
from mintpy.cli import view
from mintpy.utils import readfile

plt.close()

cfg = readfile.read_template(project_dir / 'smallbaselineApp.cfg')
ref_la, ref_lo = [int(coord) for coord in cfg['mintpy.reference.lalo'].strip("[]").split(', ')]
coh_avg = project_dir / 'avgSpatialCoh.h5'
view.main(f'{coh_avg} --pts-lalo {ref_la} {ref_lo} --noverbose --figsize 8 8'.split())

By zooming in on this plot, we can see that our reference point has a coherence value of 0.8 - that should be good enough.

#### 3.3.3 Estimate Velocity

Having checked that our network and reference point are good, we will run the remainder of the analysis. This includes performing the time series inversion, the tropospheric correction, and the topographic residual correction. If you want to read more about MintPy's SBAS workflow, you can read the paper describing the MintPy workflow [here](https://doi.org/10.1016/j.cageo.2019.104331).

In [ ]:
!smallbaselineApp.py $mintpy_config --dir $project_dir --start quick_overview

Congrats - you have successfully completed a time series InSAR analysis! Use the interactive plot below to view Mt. Edgecumbe's deformation history at specific locations.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
from mintpy.cli import tsview

plt.close()

timeseries = project_dir / 'timeseries_ERA5_demErr.h5'
cmd = f'{timeseries} --figsize 9 3 --noverbose'
tsview.main(cmd.split())

**Remember to run MintPy for all three stacks before moving on!**

## 4. (Homework) Additional Tracks

### 4.1 Running the Ascending Tracks

This notebook has walked you through performing an SBAS analysis for one InSAR stack. To fully recreate [Figure 2 from Grapenthin et al., 2022](https://doi.org/10.1029/2022GL099464) however, you'll also need to perform the same analysis for Ascending track 50 and 79.

After this course, we recommend that you also try processing these stacks. You can do this by changing the `project_name` and `scene_pair_path` variables in Section 1 to the values for the Ascending 50/79 tracks, then re-running the notebook through the end of the MintPy analysise (the end of Section 3) for both additional tracks.

Once you're done, come back to this section to re-create Grapenthin et al.'s Figure 2!

### 4.2 Recreating Grapenthin et al.'s Figure 2

Now that you have **run MintPy for all three stacks**, let's take a look at the results. Using the final velocity files from each stack, we can recreate the main portion of [Figure 2 from Grapenthin et al., 2022](https://doi.org/10.1029/2022GL099464).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from mintpy.utils import readfile

plt.close()

ascending_50, _ = readfile.read('edgecumbe_ascending_50/velocity.h5')
ascending_50 = np.ma.masked_equal(ascending_50, 0)

ascending_79, _ = readfile.read('edgecumbe_ascending_79/velocity.h5')
ascending_79 = np.ma.masked_equal(ascending_79, 0)

descending_174, _ = readfile.read('edgecumbe_descending_174/velocity.h5')
descending_174 = np.ma.masked_equal(descending_174, 0)

f, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize = (12, 6))
ax1.imshow(ascending_79, cmap='jet', vmin=-0.04, vmax=0.07)
ax1.set(title='Ascending 79')
ax2.imshow(ascending_50, cmap='jet', vmin=-0.04, vmax=0.07)
ax2.set(title='Ascending 50')
cbar_plot = ax3.imshow(descending_174, cmap='jet', vmin=-0.04, vmax=0.07)
ax3.set(title='Descending 174')

cax = ax3.inset_axes([1.05, 0.25, 0.05, 0.5])
f.colorbar(cbar_plot, ax=ax3, cax=cax, orientation='vertical')

plt.setp(plt.gcf().get_axes(), xticks=[], yticks=[])
plt.tight_layout()

CONGRATULATIONS - you have just performed a publication-quality time series InSAR analysis that is very similar to Grapenthin Et Al.'s result!!

<img src="figures/grapenthin_et_al_figure.png" alt="Grapenthin et al. LOS Velocity Plot" style="width: 1200px;"/>